# Reescrever a lista aqui para ficar mais organizado

# Ciência de Dados - Lista 1

* **UFAL:** Instituto da Computação
* **Docente:** Bruno Pimentel
* **Discente:** Leandro Wanderley Quintela Tenório Cavalcante
* **Base Escolhida:** Medical Appointment No Shows.csv

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Carregamento da base [cite: 13]
path = 'Medical Appointment No Shows.csv'
df_raw = pd.read_csv(path)

df_raw.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## Questão 1

**Problema Computacional:** Classificação Binária Supervisionada.

**Justificativa:** O objetivo é prever se um paciente irá faltar (`No-show: Yes`) ou comparecer (`No-show: No`) à consulta.

In [4]:
def gerar_diagnostico(df):
    print(f"{' DIAGNÓSTICO INICIAL ':^50}")
    print("-" * 50)
    print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
    
    info = pd.DataFrame({
        'Tipo': df.dtypes,
        'Ausentes': df.isnull().sum(),
        'Cardinalidade': df.nunique(),
        '%_Nulos': (df.isnull().sum() / len(df)) * 100
    })
    display(info)
    print(f"\nTotal de duplicados: {df.duplicated().sum()}")

gerar_diagnostico(df_raw)

               DIAGNÓSTICO INICIAL                
--------------------------------------------------
Linhas: 110527 | Colunas: 14


,Tipo,Ausentes,Cardinalidade,%_Nulos
PatientId,float64,0,62299,0.0
AppointmentID,int64,0,110527,0.0
Gender,str,0,2,0.0
ScheduledDay,str,0,103549,0.0
AppointmentDay,str,0,27,0.0
Age,int64,0,104,0.0
Neighbourhood,str,0,81,0.0
Scholarship,int64,0,2,0.0
Hipertension,int64,0,2,0.0
Diabetes,int64,0,2,0.0



Total de duplicados: 0


## Questão 2
Identifiquei a necessidade de converter datas, tratar idades negativas e verificar categorias raras.

In [5]:
def auditar_dados(df):
    print("=== Relatório de Auditoria ===")
    # 1. Idades impossíveis
    print(f"Idades negativas encontradas: {df[df['Age'] < 0].shape[0]}")
    # 2. Verificação de datas
    print(f"Tipos de data atuais: {df['ScheduledDay'].dtype}, {df['AppointmentDay'].dtype}")
    # 3. Outliers em Age
    print(f"Idade máxima: {df['Age'].max()}")

auditar_dados(df_raw)

=== Relatório de Auditoria ===
Idades negativas encontradas: 1
Tipos de data atuais: str, str
Idade máxima: 115
